In [1]:
!pip -q install langchain-groq
!pip -q install -U langchain_community tiktoken langchainhub
!pip -q install -U langchain langgraph
!pip -q install -U langchain langchain-community langchainhub
!pip -q install langchain-chroma bs4
!pip -q install huggingface_hub unstructured sentence_transformers
!pip -q install langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.2/437.2 kB 11.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 493.6 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Pre

In [2]:
import os
from pprint import pprint
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")


In [3]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="microsoft/graphcodebert-base")

2025-04-30 20:58:28.443316: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746046708.728553      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746046708.807847      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [4]:
import json
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma


json_filepath = '/kaggle/input/bash-commands/bash_commands_full.json'


with open(json_filepath, 'r', encoding='utf-8') as f:
    command_data = json.load(f)


docs = []
for item in command_data:
    docs.append(Document(
        page_content=item['description'],
        metadata={'command': item['command']}
    ))


docs = [doc for doc in docs if 'file' in doc.page_content.lower().split()]


splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=300)
chunked_docs = splitter.split_documents(docs)


chroma_db = Chroma.from_documents(
    documents=chunked_docs,
    collection_name='rag_linux_commands',
    embedding=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory="./linux_cmd_db"
)

chunked_docs[:3]


[Document(metadata={'command': 'addr2line'}, page_content='Used to convert addresses into file names and line numbers.'),
 Document(metadata={'command': 'autoupdate'}, page_content='Update a configure.in file to newer autoconf.'),
 Document(metadata={'command': 'bzip2'}, page_content='A block-sorting file compressor used to shrink given files.')]

In [5]:
retriever = chroma_db.as_retriever(search_kwargs={"k": 5})

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

In [7]:
from langchain_groq import ChatGroq

GROQ_LLM = ChatGroq(
            model="llama3-70b-8192",
        )

In [8]:

rag_prompt = PromptTemplate(
    template="""
You are an assistant for answering questions about Bash commands.
Use the provided context from command descriptions to answer the question.
If you don’t know the answer, just say “I don’t know.”
Return just your code no extra word just the code.

QUESTION: {question}

CONTEXT:
{context}

Answer:
""",
    input_variables=["question", "context"],
)

# Build the chain
rag_prompt_chain = rag_prompt | GROQ_LLM | StrOutputParser()

# Example usage
QUESTION = "Used to convert addresses into file names and line numbers"
CONTEXT = retriever.invoke(QUESTION)


result = rag_prompt_chain.invoke({"question": QUESTION, "context": CONTEXT})

print("Answer:", result)

Answer: addr2line


In [9]:
rag_chain = (
    {"context": retriever , "question": RunnablePassthrough()}
    | rag_prompt
    | GROQ_LLM
    | StrOutputParser()
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

# UTILS

In [11]:

def write_markdown_file(content, filename):
    """Writes the given content as a markdown file to the local directory.

    Args:
        content: The content to write to the file (string, list, dict, or StringPromptValue).
        filename: The filename to save the file as (without extension).
    """
    # Convert LangChain's StringPromptValue (or similar) to plain string
    if hasattr(content, "to_string"):
        content = content.to_string()
    # Fallback for any non-str object: use str()
    elif not isinstance(content, str):
        if isinstance(content, dict):
            # Render dict as key: value lines
            content = "\n".join(f"{k}: {v}" for k, v in content.items())
        elif isinstance(content, list):
            # Render list of strings as newline-separated
            content = "\n".join(str(item) for item in content)
        else:
            # Generic fallback
            content = str(content)

    # Write out the markdown file
    with open(f"{filename}.md", "w", encoding="utf-8") as f:
        f.write(content)



In [13]:
bash_code_prompt = PromptTemplate(
    template="""
You are a Bash scripting expert. 
When given a description of a task, output only the Bash code (no commentary) that accomplishes it.
Make the script robust: include comments, error-checking where appropriate, and use best practices.

TASK DESCRIPTION:
{initial_prompt}

# YOUR BASH SCRIPT:
""",
    input_variables=["initial_prompt"],
)

# Build the chain
bash_code_generator = bash_code_prompt | GROQ_LLM | StrOutputParser()

# Example usage
TASK = "List all files in the current directory (including hidden), sort them by modification time descending, and save the output to a file named files.txt. Exit with error if the directory is not accessible."

script = bash_code_generator.invoke({"initial_prompt": TASK})
print(script)


```
#!/bin/bash

# Check if the current directory is accessible
if [ ! -r "." ]; then
  echo "Error: Unable to read current directory." >&2
  exit 1
fi

# List all files in the current directory (including hidden), sort them by modification time descending
ls -a --time-style=+%s | sort -rn > files.txt
```


In [14]:
# **Modified:** Instruct model to use strict JSON format with double quotes.
research_router_prompt = PromptTemplate(
    template="""
You are an assistant routing a Bash task. Decide if you should generate code or request more info.
Return ONLY a JSON object with a single key "router_decision" (value `"generate_code"` or `"request_clarification"`), in valid JSON format (use double quotes). No extra text.

TASK DESCRIPTION:
{initial_prompt}

TASK CATEGORY:
{prompt_category}
""",
    input_variables=["initial_prompt", "prompt_category"],
)

research_router = research_router_prompt | GROQ_LLM | JsonOutputParser()

# Example
Prompt = "I need to recursively find all `.log` files modified in the last 7 days and compress them into a tar.gz. How do I do that?"
Prompt_category = "command_enquiry"

decision = research_router.invoke({
    "initial_prompt": Prompt,
    "prompt_category": Prompt_category
})

print(decision)  # -> {"router_decision": "generate_code"} or {"router_decision": "request_clarification"}


{'router_decision': 'generate_code'}


In [15]:
# **Modified:** Emphasize JSON output with key "questions".
search_rag_prompt = PromptTemplate(
    template="""
You are an expert at formulating internal questions for Bash tasks.
Given the TASK DESCRIPTION and TASK CATEGORY, produce up to three concise questions needed for a robust script.
Return ONLY a JSON object with key "questions" whose value is a list of strings (JSON array, no more than 3). No extra text.

TASK DESCRIPTION:
{initial_prompt}

TASK CATEGORY:
{prompt_category}
""",
    input_variables=["initial_prompt", "prompt_category"],
)


question_rag_chain = search_rag_prompt | GROQ_LLM | JsonOutputParser()

# Example
Prompt = "I need to back up all `.conf` files from /etc recursively, but exclude any files larger than 1MB."
prompt_category = "command_enquiry"

result = question_rag_chain.invoke({
    "initial_prompt": Prompt,
    "prompt_category": prompt_category
})

print(result)  # -> {"questions": ["Should the backup preserve directory structure?", "Do we include hidden .conf files?", "..."]}


{'questions': ['How to use find command to search for .conf files recursively in /etc?', 'How to exclude files larger than 1MB using find command?', 'How to use xargs or exec option with find command to copy files to backup location?']}


In [16]:
# **Modified:** Return the Bash code inside the JSON under "script_md" to ensure a single valid JSON block.
draft_writer_prompt = PromptTemplate(
    template="""
You are a Bash scripting assistant. Based on the TASK DESCRIPTION, TASK CATEGORY, and RESEARCH_INFO:

- If TASK_CATEGORY is "command_enquiry", produce a ready-to-run Bash script.
- Otherwise, produce a single clarification question.

**Output format**: Return ONLY one JSON object. If returning a script, the JSON should have a key "script_md" whose value is the entire script (including any ```bash code fences) as a string. If asking for clarification, use key "clarification" with the question string as value. Use strict JSON (double quotes) and no extra text.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

RESEARCH_INFO:
{research_info}
""",
    input_variables=["initial_prompt", "prompt_category", "research_info"],
)



draft_writer_chain = draft_writer_prompt | GROQ_LLM | JsonOutputParser()

# Example usage
prompt = "Archive all .log files older than 30 days in /var/log and email me if any errors occur."
prompt_category = "command_enquiry"
research_info = "Use find with -mtime +30, tar with exit-code check, then send mail via mailx."

response = draft_writer_chain.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "research_info": research_info
})

print(response)  # -> {"script_md": ""} followed by the fenced code block


{'script_md': "```bash\n#!/bin/bash\n\n# Find and archive .log files older than 30 days\nfind /var/log -name '*.log' -mtime +30 -exec tar -cf /var/log/archived_logs.tar {} +\n\n# Check if tar command was successful\nif [ $? -ne 0 ]; then\n  echo 'Error archiving log files' | mailx -s 'Error: Log File Archiving' your_email@example.com\n  exit 1\nfi\n\n# Remove the archived log files\nfind /var/log -name '*.log' -mtime +30 -delete\n```"}


In [17]:
# Rewrite Router Prompt — decides whether the generated script/clarification is sufficient
# **Modified:** Instruct JSON output with 'router_decision' in valid JSON format.
rewrite_router_prompt = PromptTemplate(
    template="""
You are an expert evaluating Bash scripts or questions for a task.
Compare the TASK_DESCRIPTION and TASK_CATEGORY to the GENERATED_OUTPUT.
Return ONLY a JSON object with key "router_decision" whose value is either "no_rewrite" or "rewrite". Use valid JSON (double quotes) and no extra text.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

GENERATED_OUTPUT:
{draft_code}
""",
    input_variables=["initial_prompt", "prompt_category", "draft_code"],
)

rewrite_router = rewrite_router_prompt | GROQ_LLM | JsonOutputParser()

# Example
prompt = "Archive all .log files older than 7 days under /var/log and report failures via email."
prompt_category = "command_enquiry"
draft_code = """#!/usr/bin/env bash
find /var/log -name '*.log' -mtime +7 | xargs tar -czf logs.tar.gz
# Missing email notification logic
"""

decision = rewrite_router.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "draft_code": draft_code
})

print(decision)


{'router_decision': 'rewrite'}


In [18]:
# **Modified:** Return JSON with key "draft_analysis" in valid JSON.
draft_analysis_prompt = PromptTemplate(
    template="""
You are the Quality Control Agent for Bash scripting tasks.
Read TASK_DESCRIPTION, TASK_CATEGORY, and RESEARCH_INFO, and analyze the GENERATED_OUTPUT (script or question).
Return ONLY a JSON object with key "draft_analysis" and the feedback string as value. Use valid JSON (double quotes), no extra text.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

RESEARCH_INFO:
{research_info}

GENERATED_OUTPUT:
{draft_code}
""",
    input_variables=["initial_prompt", "prompt_category", "research_info", "draft_code"],
)


draft_analysis_chain = draft_analysis_prompt | GROQ_LLM | JsonOutputParser()

# Example usage
prompt = "Archive all .log files older than 7 days under /var/log and report failures via email."
prompt_category = "command_enquiry"
research_info = "Use find with -mtime and tar; then check exit code of tar and send mail."
draft_code = """#!/usr/bin/env bash
find /var/log -name '*.log' -mtime +7 | xargs tar -czf logs.tar.gz
"""

analysis = draft_analysis_chain.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "research_info": research_info,
    "draft_code": draft_code
})

print(analysis)  # -> {"draft_analysis": "...actionable feedback..."}


{'draft_analysis': "The script is almost correct, but it's missing the part to report failures via email. Also, it's not checking the exit code of tar. Consider adding a check for the exit code of tar and sending an email if it's non-zero. Additionally, the script should handle cases where no files are found by find."}


In [19]:
# **Modified:** Return the improved script under "final_output" in valid JSON.
rewrite_script_prompt = PromptTemplate(
    template="""
You are the Final Bash Script Agent. Using the QC feedback, rewrite the draft Bash script to fully meet the TASK_DESCRIPTION.
Return ONLY a JSON object with key "final_output" whose value is the improved script (or clarification question) as a string. Use valid JSON (double quotes) and no extra text.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

RESEARCH_INFO:
{research_info}

DRAFT_OUTPUT:
{draft_code}

QC_FEEDBACK:
{code_analysis}
""",
    input_variables=["initial_prompt", "prompt_category", "research_info", "draft_code", "code_analysis"],
)



rewrite_chain = rewrite_script_prompt | GROQ_LLM | JsonOutputParser()

# Example usage
prompt = "Archive all .log files older than 7 days under /var/log and notify me if compression fails."
prompt_category = "command_enquiry"
research_info = "Use find with -mtime +7, tar with exit-code check, then send mail via mailx."
draft_script = """#!/usr/bin/env bash
find /var/log -name '*.log' -mtime +7 | xargs tar -czf logs.tar.gz
"""

qc_feedback = "The script compresses files but lacks error handling and email notification."

result = rewrite_chain.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "research_info": research_info,
    "draft_code": draft_script,
    "code_analysis": qc_feedback
})

print(result["final_output"])
# -> Improved Bash script with proper error checks and mailx notification


#!/usr/bin/env bash

ARCHIVE_FILE=logs_$(date '+%Y-%m-%d').tar.gz

find /var/log -name '*.log' -mtime +7 | tar -czf $ARCHIVE_FILE -T -
if [ $? -ne 0 ]; then
    echo 'Compression failed.' | mailx -s 'Compression failed' your_email@example.com
fi


In [20]:
from langchain.schema import Document
from langgraph.graph import END, StateGraph

In [21]:
from typing_extensions import TypedDict
from typing import List

### State

class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        initial_prompt: email
        prompt_category: email category
        draft_code: LLM generation
        final_code: LLM generation
        research_info: list of documents
        info_needed: whether to add search info
        num_steps: number of steps
    """
    initial_prompt : str
    prompt_category : str
    draft_code : str
    final_code : str
    research_info : List[str] # this will now be the RAG results
    info_needed : bool
    num_steps : int
    draft_code_feedback : dict
    rag_questions : List[str]

In [22]:
def categorize_prompt(state):
    """take the initial prompt and categorize it"""
    print("---CATEGORIZING INITIAL PROMPT---")
    initial_prompt = state['initial_prompt']
    num_steps = int(state['num_steps'])
    num_steps += 1

    prompt_category = bash_code_prompt.invoke({"initial_prompt": initial_prompt})
    print(prompt_category)
    # save to local disk
    write_markdown_file(prompt_category, "prompt_category")

    return {"prompt_category": prompt_category, "num_steps":num_steps}

In [23]:
def research_info_search(state):

    print("---RESEARCH INFO RAG---")
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    num_steps = state['num_steps']
    num_steps += 1

    # Web search
    questions = question_rag_chain.invoke({"initial_prompt": initial_prompt,
                                            "prompt_category": prompt_category })
    questions = questions['questions']
    # print(questions)
    rag_results = []
    for question in questions:
        print(question)
        temp_docs = rag_chain.invoke(question)
        print(temp_docs)
        question_results = question + '\n\n' + temp_docs + "\n\n\n"
        if rag_results is not None:
            rag_results.append(question_results)
        else:
            rag_results = [question_results]
    print(rag_results)
    print(type(rag_results))
    write_markdown_file(rag_results, "research_info")
    write_markdown_file(questions, "rag_questions")
    return {"research_info": rag_results,"rag_questions":questions, "num_steps":num_steps}

In [24]:
def draft_email_writer(state):
    print("---DRAFT CODE WRITER---")
    # Get inputs from state
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    research_info = state["research_info"]
    num_steps = state["num_steps"]
    num_steps += 1

    # Invoke the draft writer chain
    draft_code = draft_writer_chain.invoke({
        "initial_prompt": initial_prompt,
        "prompt_category": prompt_category,
        "research_info": research_info
    })
    print(draft_code)

    # **Modified:** Use 'script_md' key if present (chain returns {'script_md': ...})
    code_draft = draft_code.get("script_md", draft_code.get("draft_code", ""))
    write_markdown_file(code_draft, "draft_code")

    return {"draft_code": code_draft, "num_steps": num_steps}


In [25]:
def analyze_draft_email(state):
    print("---DRAFT CODE ANALYZER---")
    # Get the state
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    draft_code = state["draft_code"]
    research_info = state["research_info"]
    num_steps = state["num_steps"]
    num_steps += 1

    # Invoke the draft analysis chain
    analysis_dict = draft_analysis_chain.invoke({
        "initial_prompt": initial_prompt,
        "prompt_category": prompt_category,
        "research_info": research_info,
        "draft_code": draft_code
    })

    # **Modified:** Extract just the feedback string
    feedback_str = analysis_dict.get("draft_analysis", "")
    write_markdown_file(feedback_str, "draft_code_feedback")

    return {"draft_code_feedback": feedback_str, "num_steps": num_steps}


In [31]:
def rewrite_email(state):
    print("---REWRITE CODE---")
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    draft_code = state["draft_code"]
    research_info = state["research_info"]
    draft_code_feedback = state["draft_code_feedback"]

    try:
        result = rewrite_chain.invoke({
            "initial_prompt": initial_prompt,
            "prompt_category": prompt_category,
            "research_info": research_info,
            "draft_code": draft_code,
            "code_analysis": draft_code_feedback
        })
    except Exception:
        print("Final rewrite JSON parse failed, using raw fallback.")
        raw = (rewrite_script_prompt | GROQ_LLM | StrOutputParser()).invoke({
            "initial_prompt": initial_prompt,
            "prompt_category": prompt_category,
            "research_info": research_info,
            "draft_code": draft_code,
            "code_analysis": draft_code_feedback
        })
        try:
            result = json.loads(raw)
        except:
            result = {"final_output": ""}
    # **Modified:** Extract the final script text from result
    final_output = result.get("final_output", "")
    write_markdown_file(final_output, "final_code")
    return {"final_code": final_output}


In [32]:
def no_rewrite(state):
    print("---NO REWRITE CODE ---")
    ## Get the state
    draft_code = state["draft_code"]
    num_steps = state['num_steps']
    num_steps += 1

    write_markdown_file(str(draft_code), "final_code")
    return {"final_code": draft_code, "num_steps":num_steps}

In [33]:
def state_printer(state):
    """print the state"""
    print("---STATE PRINTER---")
    print(f"Initial Prompt: {state['initial_prompt']} \n" )
    print(f"Prompt Category: {state['prompt_category']} \n")
    print(f"Draft Code: {state['draft_code']} \n" )
    print(f"Final Code: {state['final_code']} \n" )
    print(f"Research Info: {state['research_info']} \n")
    print(f"RAG Questions: {state['rag_questions']} \n")
    print(f"Num Steps: {state['num_steps']} \n")
    return

# Conditional Edges

In [34]:
def route_to_research(state):
    print("---ROUTE TO RESEARCH---")
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]

    try:
        # Attempt JSON parsing as usual
        router = research_router.invoke({"initial_prompt": initial_prompt, "prompt_category": prompt_category})
    except Exception as e:
        # **Modified:** Fallback to raw string on JSON parse failure
        print("Router JSON parse failed, using raw fallback.")
        raw = (research_router_prompt | GROQ_LLM | StrOutputParser()).invoke({"initial_prompt": initial_prompt, "prompt_category": prompt_category})
        try:
            router = json.loads(raw)
        except Exception as parse_err:
            # last resort default
            router = {"router_decision": "request_clarification"}
    print(router)
    decision = router.get("router_decision", "request_clarification")
    print(decision)
    if decision == 'request_clarification':
        return "research_info"
    else:
        return "draft_code"


In [35]:
def route_to_rewrite(state):
    print("---ROUTE TO REWRITE---")
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    draft_code = state["draft_code"]

    try:
        router = rewrite_router.invoke({
            "initial_prompt": initial_prompt,
            "prompt_category": prompt_category,
            "draft_code": draft_code
        })
    except Exception:
        print("Rewrite router JSON parse failed, using raw fallback.")
        raw = (rewrite_router_prompt | GROQ_LLM | StrOutputParser()).invoke({
            "initial_prompt": initial_prompt,
            "prompt_category": prompt_category,
            "draft_code": draft_code
        })
        try:
            router = json.loads(raw)
        except:
            router = {"router_decision": "no_rewrite"}
    print(router)
    return router.get("router_decision", "no_rewrite")


In [36]:

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("categorize_prompt", categorize_prompt) # categorize email
workflow.add_node("research_info_search", research_info_search) # web search
workflow.add_node("state_printer", state_printer)
workflow.add_node("draft_email_writer", draft_email_writer)
workflow.add_node("analyze_draft_email", analyze_draft_email)
workflow.add_node("rewrite_email", rewrite_email)
workflow.add_node("no_rewrite", no_rewrite)



In [37]:
workflow.set_entry_point("categorize_prompt")


workflow.add_edge("categorize_prompt", "research_info_search")
workflow.add_edge("research_info_search", "draft_email_writer")


workflow.add_conditional_edges(
    "draft_email_writer",
    route_to_rewrite,
    {
        "rewrite": "analyze_draft_email",
        "no_rewrite": "no_rewrite",
    },
)
workflow.add_edge("analyze_draft_email", "rewrite_email")
workflow.add_edge("no_rewrite", "state_printer")
workflow.add_edge("rewrite_email", "state_printer")
workflow.add_edge("state_printer", END)

In [38]:
# Compile
app = workflow.compile()

In [40]:
QUERY = "Count md5sum of all '*.py' files in /testbed folder with subfolders."
# run the agent
inputs = {"initial_prompt": QUERY, "num_steps":0}
for output in app.stream(inputs):
    for key, value in output.items():
        pprint(f"Finished running: {key}:")

---CATEGORIZING INITIAL PROMPT---
text="\nYou are a Bash scripting expert. \nWhen given a description of a task, output only the Bash code (no commentary) that accomplishes it.\nMake the script robust: include comments, error-checking where appropriate, and use best practices.\n\nTASK DESCRIPTION:\nCount md5sum of all '*.py' files in /testbed folder with subfolders.\n\n# YOUR BASH SCRIPT:\n"
'Finished running: categorize_prompt:'
---RESEARCH INFO RAG---
Should the script only consider '*.py' files in the '/testbed' folder and its subfolders, or should it also consider files with '.py' extension in the subfolders' subfolders?
find /testbed -name "*.py"
What should the script do if it encounters a '*.py' file with no read permission?
I don’t know.
Should the script output the md5sum count in a specific format, such as a table or a single line?
I don’t know.
['Should the script only consider \'*.py\' files in the \'/testbed\' folder and its subfolders, or should it also consider files wit